## EMD Demo

[EnergyFlow website](https://energyflow.network)

In this tutorial, we demonstrate how to compute EMD values for particle physics events. The core of the computation is done using the [Python Optimal Transport](https://pot.readthedocs.io) library with EnergyFlow providing a convenient interface to particle physics events. Batching functionality is also provided using the builtin multiprocessing library to distribute computations to worker processes.

### Energy Mover's Distance

The Energy Mover's Distance was introduced in [1902.02346](https://arxiv.org/abs/1902.02346) as a metric between particle physics events. Closely related to the Earth Mover's Distance, the EMD solves an optimal transport problem between two distributions of energy (or transverse momentum), and the associated distance is the "work" required to transport supply to demand according to the resulting flow. Mathematically, we have
$$\text{EMD}(\mathcal E, \mathcal E') = \min_{\{f_{ij}\ge0\}}\sum_{ij} f_{ij} \frac{\theta_{ij}}{R} + \left|\sum_i E_i - \sum_j E'_j\right|,$$
$$\sum_{j} f_{ij} \le E_i,\,\,\, \sum_i f_{ij} \le E'_j,\,\,\,\sum_{ij}f_{ij}= \min\Big(\sum_iE_i,\,\sum_jE'_j\Big).$$

### Imports

In [ ]:
import numpy as np
%load_ext wurlitzer
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

import awkward as ak
import energyflow as ef

### Plot Style

In [ ]:
import shutil

plt.rcParams['figure.figsize'] = (4,4)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'serif'

# Use TeX rendering only when a LaTeX executable is available.
plt.rcParams['text.usetex'] = shutil.which('latex') is not None

### Load Zjet and TTbar Point Samples

In [ ]:
# load point-level event collections [pt, eta, phi] with zero padding
zjet_allpoints = np.load('/oscar/data/mleblan6/rjain/ppzjj_100k/hadronization_points.npy')
ttbar_allpoints = np.load('/oscar/data/mleblan6/lhay/ttbar_100k/hadronization_ttbar_points.npy')

num = 750

# jet radius used throughout the EMD examples
R = 0.4

def build_event_list(point_array, max_events, center_and_clip=False):
    """Convert padded point tensors into EnergyFlow-style [pt, y, phi] arrays per event."""
    events = []
    for event_points in point_array:
        if len(events) >= max_events:
            break

        # keep only physical entries (padding has pt == 0)
        x = np.asarray(event_points, dtype=np.float64)
        if x.ndim != 2 or x.shape[1] != 3:
            continue

        x = x[np.isfinite(x).all(axis=1)]
        x = x[x[:, 0] > 0]
        if x.shape[0] == 0:
            continue

        if center_and_clip:
            yphi_avg = np.average(x[:, 1:3], weights=x[:, 0], axis=0)
            x[:, 1:3] -= yphi_avg
            x = x[np.linalg.norm(x[:, 1:3], axis=1) <= R]
            if x.shape[0] == 0:
                continue

        events.append(x)

    return events

Zjets = build_event_list(zjet_allpoints, num, center_and_clip=False)
TTbars = build_event_list(ttbar_allpoints, num, center_and_clip=False)

print(f'Loaded {len(Zjets)} Zjet events and {len(TTbars)} TTbar events.')
print(f'Average points/event (Zjet): {np.mean([ev.shape[0] for ev in Zjets]):.2f}')
print(f'Average points/event (TTbar): {np.mean([ev.shape[0] for ev in TTbars]):.2f}')

### Event Display with EMD Flow

In [ ]:
# choose interesting events from the Zjet sample
ev0, ev1 = Zjets[0], TTbars[0]
print(ev0, ev1)
print(len(Zjets))
# calculate the EMD and the optimal transport flow
R = 4
emdval, G = ef.emd.emd(ev0, ev1, R=R, return_flow=True)

# plot the two events
colors = ['teal', 'darkred']
labels = ['TTbar event', 'Zjet event']
for i, ev in enumerate([ev0, ev1]):
    pts, ys, phis = ev[:, 0], ev[:, 1], ev[:, 2]
    plt.scatter(ys, phis, marker='o', s=2 * pts, color=colors[i], lw=0, zorder=10, label=labels[i])
    
# plot the flow
mx = G.max()
xs, xt = ev0[:, 1:3], ev1[:, 1:3]
for i in range(xs.shape[0]):
    for j in range(xt.shape[0]):
        if G[i, j] > 0:
            plt.plot([xs[i, 0], xt[j, 0]], [xs[i, 1], xt[j, 1]],
                     alpha=G[i, j] / mx, lw=2, color='black')

# plot settings
plt.xlim(-R, R); plt.ylim(-R, R)
plt.xlabel('Rapidity'); plt.ylabel('Azimuthal Angle')
plt.xticks(np.linspace(-R, R, 5)); plt.yticks(np.linspace(-R, R, 5))

plt.text(0.6, 0.03, 'EMD: {:.1f} GeV'.format(emdval), fontsize=10, transform=plt.gca().transAxes)
plt.legend(loc=(0.1, 1.0), frameon=False, ncol=2, handletextpad=0)

plt.show()

In [ ]:
# choose interesting events from the Zjet sample
ev0, ev1 = Zjets[0], Zjets[1]
print(ev0, ev1)
print(len(Zjets))
# calculate the EMD and the optimal transport flow
R = 4
emdval, G = ef.emd.emd(ev0, ev1, R=R, return_flow=True)

# plot the two events
colors = ['teal', 'darkred']
labels = [r'$Zjj$ event 0', r'$Zjj$ event 1']
for i, ev in enumerate([ev0, ev1]):
    pts, ys, phis = ev[:, 0], ev[:, 1], ev[:, 2]
    plt.scatter(ys, phis, marker='o', s=2 * pts, color=colors[i], lw=0, zorder=10, label=labels[i])
    
# plot the flow
mx = G.max()
xs, xt = ev0[:, 1:3], ev1[:, 1:3]
for i in range(xs.shape[0]):
    for j in range(xt.shape[0]):
        if G[i, j] > 0:
            plt.plot([xs[i, 0], xt[j, 0]], [xs[i, 1], xt[j, 1]],
                     alpha=G[i, j] / mx, lw=2, color='black')

# plot settings
plt.xlim(-R, R); plt.ylim(-R, R)
plt.xlabel('Rapidity'); plt.ylabel('Azimuthal Angle')
plt.xticks(np.linspace(-R, R, 5)); plt.yticks(np.linspace(-R, R, 5))

plt.text(0.6, 0.03, 'EMD: {:.1f} GeV'.format(emdval), fontsize=10, transform=plt.gca().transAxes)
plt.legend(loc=(0.1, 1.0), frameon=False, ncol=2, handletextpad=0)

plt.show()

In [ ]:
from pyspecter.SPECTER import SPECTER

# SPECTER expects fixed-shape batches, so use the padded point tensors directly.
specter = SPECTER(compile=True)

ev0_spec = zjet_allpoints[0][None, :, :]
ev1_spec = ttbar_allpoints[0][None, :, :]

# Spectral EMD needs omega_max for unbalanced events.
semd = specter.spectralEMD(ev0_spec, ev1_spec, omega_max=4.2)[0]

fig, ax = plt.subplots(figsize=(5, 4))
colors = ['teal', 'orange']
labels = ['Zjet event', 'TTbar event']

for i, ev in enumerate([ev0_spec[0], ev1_spec[0]]):
    valid = ev[:, 0] > 0
    pts, ys, phis = ev[valid, 0], ev[valid, 1], ev[valid, 2]
    ax.scatter(ys, phis, marker='o', s=2 * pts, color=colors[i], lw=0, zorder=10, label=labels[i])

ax.set_xlabel('Rapidity')
ax.set_ylabel('Azimuthal Angle')
# ax.set_title('Spectral EMD with SPECTER', pad=14)
ax.text(0.60, 0.03, f'sEMD: {float(semd):.2e}', fontsize=10, transform=ax.transAxes)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.12), frameon=False, ncol=2, handletextpad=0.4)
fig.tight_layout()
plt.show()

In [ ]:
# proxy diagnostic for spectral EMD: histogram in omega with spectral density on y-axis
spec0 = specter.compute_spectral_representation(ev0_spec)[0]
spec1 = specter.compute_spectral_representation(ev1_spec)[0]

# drop the bookkeeping entry at omega=0
spec0 = np.asarray(spec0)[1:]
spec1 = np.asarray(spec1)[1:]

omega0, weight0 = spec0[:, 0], spec0[:, 1]
omega1, weight1 = spec1[:, 0], spec1[:, 1]

omega_max = max(np.max(omega0), np.max(omega1))
bins = np.linspace(0, omega_max, 70)

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(omega0, bins=bins, weights=weight0, density=True, histtype='step', linewidth=2, color='teal', label='Zjet')
ax.hist(omega1, bins=bins, weights=weight1, density=True, histtype='step', linewidth=2, color='orange', label='TTbar')

ax.set_xlabel(r'$\omega$')
ax.set_ylabel('Spectral density')
ax.legend(loc='upper right', frameon=False)
fig.tight_layout()
plt.show()

### Intrinsic Dimension of Zjet and TTbar Events

The correlation dimension of a dataset is a type of fractal dimension which quantifies the dimensionality of the space of events at different energy scales $Q$.

It is motivated by the fact that the number of neighbors a point has in a ball of radius $Q$ grows as $Q^\mathrm{dim}$, giving rise to the definition:

$$ \dim (Q) = Q\frac{\partial}{\partial Q} \ln \sum_{i<j} \Theta(\mathrm{EMD}(\mathcal E_i, \mathcal E_j) < Q).$$

In [ ]:
# # compute pairwise EMDs between all events (takes about 3 minutes, can change n_jobs if you have more cores)
# z_emds = ef.emd.emds(Zjets, R=R, norm=True, verbose=1, n_jobs=-1, print_every=25000)
# t_emds = ef.emd.emds(TTbars, R=R, norm=True, verbose=1, n_jobs=-1, print_every=25000)

In [ ]:
# # prepare for histograms
# bins = 10**np.linspace(-2, 0, 60)
# reg = 10**-30
# midbins = (bins[:-1] + bins[1:]) / 2
# dmidbins = np.log(midbins[1:]) - np.log(midbins[:-1]) + reg
# midbins2 = (midbins[:-1] + midbins[1:]) / 2

# # compute the correlation dimensions
# dims = []
# for emd_vals in [z_emds, t_emds]:
#     uemds = np.triu(emd_vals)
#     counts = np.cumsum(np.histogram(uemds[uemds > 0], bins=bins)[0])
#     dims.append((np.log(counts[1:] + reg) - np.log(counts[:-1] + reg)) / dmidbins)

In [ ]:
# plot the correlation dimensions
plt.plot(midbins2, dims[0], '-', color='blue', label='Zjets')
plt.plot(midbins2, dims[1], '-', color='red', label='TTbar')

# labels
plt.legend(loc='center right', frameon=False)

# plot style
plt.xscale('log')
plt.xlabel('Energy Scale Q/pT'); plt.ylabel('Correlation Dimension')
plt.xlim(0.02, 1); plt.ylim(0, 5)

plt.show()

## Using Built-In Correlation Dimension

In [ ]:
# create external EMD handlers that will compute the correlation dimensions on the fly
zcorrdim = ef.emd.wasserstein.CorrelationDimension(60, 0.01, 1)
tcorrdim = ef.emd.wasserstein.CorrelationDimension(60, 0.01, 1)

# compute pairwise EMDs between all events (takes about 3 minutes, can change n_jobs if you have more cores)
ef.emd.emds(Zjets, R=R, norm=True, verbose=1, n_jobs=-1, print_every=-10, external_emd_handler=zcorrdim)
ef.emd.emds(TTbars, R=R, norm=True, verbose=1, n_jobs=-1, print_every=-10, external_emd_handler=tcorrdim)

In [ ]:
# plot the correlation dimensions
plt.plot(zcorrdim.corrdim_bins(), zcorrdim.corrdims()[0], '-', color='blue', label='Zjets')
plt.plot(tcorrdim.corrdim_bins(), tcorrdim.corrdims()[0], '-', color='red', label='TTbar')

# labels
plt.legend(loc='center right', frameon=False)

# plot style
plt.xscale('log')
plt.xlabel('Energy Scale Q/pT'); plt.ylabel('Correlation Dimension')
plt.xlim(0.02, 1); plt.ylim(0, 5)

plt.show()

In [ ]:
# sanity-check current build_event_list output nesting
print('type(Zjets):', type(Zjets), 'len:', len(Zjets))
print('type(Zjets[0]):', type(Zjets[0]), 'shape:', Zjets[0].shape)
print('first 10 event sizes:', [ev.shape[0] for ev in Zjets[:10]])
print('min/max event size in first 200:', min(ev.shape[0] for ev in Zjets[:200]), max(ev.shape[0] for ev in Zjets[:200]))